In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import copy

# הגדרת המכשיר (GPU אם זמין)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
# הגדרת התמרות
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

# מחלקה שעוטפת את הנתונים ומערבבת את הפיקסלים אם צריך
class PermutedMNIST(torch.utils.data.Dataset):
    def __init__(self, dataset, permutation=None):
        self.dataset = dataset
        self.permutation = permutation

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        if self.permutation is not None:
            img = img.view(-1)[self.permutation].view(1, 28, 28)
        return img, label

# הגדרת מספר המשימות
num_tasks = 10
tasks = {}

# יצירת משימות בלולאה
for i in range(num_tasks):
    # המשימה הראשונה (0) תהיה ללא ערבוב. השאר מעורבבות.
    perm = None if i == 0 else torch.randperm(28 * 28)

    tasks[i] = {
        'train': DataLoader(PermutedMNIST(train_dataset, perm), batch_size=64, shuffle=True),
        'test': DataLoader(PermutedMNIST(test_dataset, perm), batch_size=1000, shuffle=False)
    }

print(f"Permuted MNIST Data ready for {num_tasks} Tasks!")

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 500kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.57MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 16.4MB/s]

Permuted MNIST Data ready for 10 Tasks!


In [3]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # הגדרת רשת בעלת 6 שכבות (Linear Layers)
        self.fc1 = nn.Linear(28 * 28, 400)
        self.fc2 = nn.Linear(400, 400)
        self.fc3 = nn.Linear(400, 400)
        self.fc4 = nn.Linear(400, 400)
        self.fc5 = nn.Linear(400, 400)
        self.fc6 = nn.Linear(400, 10)

    def forward(self, x):
        # שינוי הממד של תמונת הקלט (שטוח לכל הפיקסלים)
        x = x.view(-1, 28 * 28)

        # מעבר דרך 5 השכבות הנסתרות עם פונקציית אקטיבציה ReLU
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = F.relu(self.fc5(x))

        # שכבת המוצא
        return self.fc6(x)

In [4]:
def compute_fisher(model, data_loader):
    fisher_matrix = {}
    for name, param in model.named_parameters():
        fisher_matrix[name] = torch.zeros_like(param.data)

    model.eval()
    for data, target in data_loader:
        data, target = data.to(device), target.to(device)
        model.zero_grad()
        output = model(data)
        # חישוב פישר אמפירי מדויק יותר עם log_softmax
        loss = F.nll_loss(F.log_softmax(output, dim=1), target)
        loss.backward()

        for name, param in model.named_parameters():
            if param.grad is not None:
                fisher_matrix[name] += param.grad.data ** 2 / len(data_loader)

    return fisher_matrix

def ewc_penalty(model, fisher_matrices, opt_weights):
    penalty = 0
    # סכימת העונש מכל המשימות הקודמות
    for task_id in fisher_matrices:
        for name, param in model.named_parameters():
            fisher = fisher_matrices[task_id][name]
            opt_w = opt_weights[task_id][name]
            penalty += (fisher * (param - opt_w) ** 2).sum()
    return penalty

def test_model(model, test_loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    return 100. * correct / len(test_loader.dataset)

In [5]:
# פרמטרים לפי המאמר (Epochs יכול להיות 20 אם יש לך זמן ריצה, 10 ייתן מגמה טובה)
epochs_per_task = 20
lr = 0.001  # הגדלנו את קצב הלמידה פי 10!
lamda = 3000

model_sgd = Net().to(device)
model_ewc = Net().to(device)

optimizer_sgd = optim.SGD(model_sgd.parameters(), lr=lr, momentum=0.9)
optimizer_ewc = optim.SGD(model_ewc.parameters(), lr=lr, momentum=0.9)

fisher_matrices = {}
opt_weights = {}

# אתחול אוטומטי של מילוני ההיסטוריה לכל המשימות
history_sgd = {i: [] for i in range(num_tasks)}
history_ewc = {i: [] for i in range(num_tasks)}

print("Models initialized. Ready to start training sequence.")

Models initialized. Ready to start training sequence.


In [ ]:
total_epochs = 0

for task_id, task_data in tasks.items():
    print(f"\n--- Training on Task {task_id+1}/{num_tasks} ---")

    for epoch in range(epochs_per_task):
        model_sgd.train()
        model_ewc.train()

        for batch_idx, (data, target) in enumerate(task_data['train']):
            data, target = data.to(device), target.to(device)

            # --- אימון SGD רגיל (ללא זיכרון) ---
            optimizer_sgd.zero_grad()
            output_sgd = model_sgd(data)
            loss_sgd = F.cross_entropy(output_sgd, target)
            loss_sgd.backward()
            optimizer_sgd.step()

            # --- אימון EWC ---
            optimizer_ewc.zero_grad()
            output_ewc = model_ewc(data)
            loss_ce = F.cross_entropy(output_ewc, target)

            # הוספת פונקציית העונש אם אנחנו אחרי משימה 0
            loss_ewc = loss_ce
            if len(fisher_matrices) > 0:
                loss_ewc += (lamda / 2) * ewc_penalty(model_ewc, fisher_matrices, opt_weights)

            loss_ewc.backward()
            optimizer_ewc.step()

        # בדיקת הדיוק לכל המשימות ושמירה לגרף
        ewc_current_accs = []
        sgd_current_accs = []
        for test_task_id in tasks.keys():
            acc_sgd = test_model(model_sgd, tasks[test_task_id]['test'])
            acc_ewc = test_model(model_ewc, tasks[test_task_id]['test'])

            history_sgd[test_task_id].append(acc_sgd)
            history_ewc[test_task_id].append(acc_ewc)

            # שומרים רק את המשימות שכבר ראינו בשביל ההדפסה
            if test_task_id <= task_id:
                ewc_current_accs.append(acc_ewc)
                sgd_current_accs.append(acc_sgd)

        # הדפסת הדיוק של כל המשימות שכבר נלמדו (מעוגל)
        ewc_str = ", ".join([f"{a:.0f}%" for a in ewc_current_accs])
        sgd_str = ", ".join([f"{a:.0f}%" for a in sgd_current_accs])

        print(f"Epoch {epoch+1:02d} | "
              f"EWC: [{ewc_str}] | "
              f"SGD: [{sgd_str}]")

    # --- בסיום כל משימה: חישוב פישר ושמירת המשקלים האופטימליים ---
    print(f"Computing Fisher Information Matrix for Task {task_id+1}...")
    fisher_matrices[task_id] = compute_fisher(model_ewc, task_data['train'])

    opt_weights[task_id] = {}
    for name, param in model_ewc.named_parameters():
        opt_weights[task_id][name] = param.data.clone()

    total_epochs += epochs_per_task


--- Training on Task 1/10 ---
Epoch 01 | EWC: [53%] | SGD: [49%]
Epoch 02 | EWC: [86%] | SGD: [85%]
Epoch 03 | EWC: [91%] | SGD: [91%]
Epoch 04 | EWC: [94%] | SGD: [93%]
Epoch 05 | EWC: [95%] | SGD: [94%]
Epoch 06 | EWC: [96%] | SGD: [95%]
Epoch 07 | EWC: [96%] | SGD: [96%]
Epoch 08 | EWC: [96%] | SGD: [96%]
Epoch 09 | EWC: [97%] | SGD: [97%]
Epoch 10 | EWC: [97%] | SGD: [97%]
Epoch 11 | EWC: [97%] | SGD: [97%]
Epoch 12 | EWC: [97%] | SGD: [97%]
Epoch 13 | EWC: [97%] | SGD: [97%]
Epoch 14 | EWC: [97%] | SGD: [98%]
Epoch 15 | EWC: [97%] | SGD: [97%]
Epoch 16 | EWC: [97%] | SGD: [98%]
Epoch 17 | EWC: [97%] | SGD: [98%]
Epoch 18 | EWC: [97%] | SGD: [98%]


In [ ]:
plt.figure(figsize=(12, 6))
x_axis = range(1, total_epochs + 1)

# שרטוט התוצאות של משימה A (משימה 0 במילון)
plt.plot(x_axis, history_ewc[0], label='EWC Task A', color='red', linestyle='-')
plt.plot(x_axis, history_sgd[0], label='SGD Task A', color='red', linestyle='--')

# שרטוט התוצאות של משימה B (משימה 1 במילון)
plt.plot(x_axis, history_ewc[1], label='EWC Task B', color='green', linestyle='-')
plt.plot(x_axis, history_sgd[1], label='SGD Task B', color='green', linestyle='--')

# שרטוט התוצאות של משימה C (משימה 2 במילון)
plt.plot(x_axis, history_ewc[2], label='EWC Task C', color='blue', linestyle='-')
plt.plot(x_axis, history_sgd[2], label='SGD Task C', color='blue', linestyle='--')

# קווים אנכיים לציון החלפת המשימות לאורך כל 10 המשימות
for i in range(1, num_tasks):
    plt.axvline(x=epochs_per_task * i, color='k', linestyle=':', alpha=0.3)

plt.title('Continual Learning: EWC vs SGD (Tracking First 3 Tasks: A, B, C)')
plt.xlabel('Training Epochs')
plt.ylabel('Test Accuracy (%)')
plt.ylim([10, 105]) # גבול תחתון הורחב כדי לראות את הנפילה של SGD לאורך זמן
plt.legend(loc='lower left')
plt.grid(True, alpha=0.4)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- עיבוד נתונים לגרף הממוצעים ---
task_ids = list(tasks.keys())
num_tasks = len(task_ids)

# נתחיל לצייר ממשימה 2 (כי אחרי משימה 1 עדיין אין "שכחה" של משימות עבר)
x_tasks = range(2, num_tasks + 1)
ewc_avg_acc = []
sgd_avg_acc = []

for i in range(1, num_tasks):
    # חישוב האינדקס של האפוק האחרון של המשימה הנוכחית
    end_epoch = (i + 1) * epochs_per_task - 1

    # המשימות שהמודל כבר ראה עד לנקודה זו (כולל)
    seen_tasks = task_ids[:i+1]

    # שליפת הדיוק של כל המשימות שכבר נלמדו, נכון לסוף המשימה הנוכחית
    ewc_accs = [history_ewc[t][end_epoch] for t in seen_tasks]
    sgd_accs = [history_sgd[t][end_epoch] for t in seen_tasks]

    # חישוב הממוצע ושמירה
    ewc_avg_acc.append(np.mean(ewc_accs))
    sgd_avg_acc.append(np.mean(sgd_accs))

# --- ציור הגרף (בסגנון גרף B מהמאמר) ---
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(x_tasks, ewc_avg_acc, marker='o', color='#c44e52', label='EWC')
ax.plot(x_tasks, sgd_avg_acc, marker='o', color='#4c72b0', label='SGD+dropout')

# קו מקווקו לביצועי משימה בודדת (כאן נניח שהמקסימום שהמודל מגיע אליו הוא סביב 98-100)
# אפשר לשנות את הערך בהתאם למה שיצא לך בגרף A
single_task_baseline = max(history_ewc[task_ids[0]])
ax.axhline(y=single_task_baseline, color='#333333', linestyle='--')

# עיצוב טקסט במקום מקרא רגיל
if len(x_tasks) > 0:
    last_x = x_tasks[-1]
    ax.text(last_x, ewc_avg_acc[-1] - 2, 'EWC', color='#c44e52', fontsize=12, ha='right', va='top')
    ax.text(last_x, sgd_avg_acc[-1] - 2, 'SGD', color='#4c72b0', fontsize=12, ha='right', va='top')
    ax.text(last_x, single_task_baseline + 1, 'single task performance', color='black', fontsize=11, ha='right', va='bottom')

# עיצוב מינימליסטי
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', direction='out')

ax.set_xlabel('Number of tasks', fontsize=12)
ax.set_ylabel('Average Fraction Correct (%)', fontsize=12)

# הגדרת ציר X כך שיציג רק מספרים שלמים של משימות
ax.set_xticks(x_tasks)

plt.tight_layout()
plt.show()

In [ ]:
# 1. פונקציה לייצור פרמוטציות חלקיות (רק אזור מסוים במרכז מעורבב)
def create_partial_permutation(size):
    perm = torch.arange(28 * 28)
    grid = perm.view(28, 28)

    start = (28 - size) // 2
    end = start + size

    # גזירת האזור המבוקש וערבוב שלו
    region = grid[start:end, start:end].flatten()
    shuffled_region = region[torch.randperm(len(region))]

    # החזרת הערכים המעורבבים למקומם
    grid[start:end, start:end] = shuffled_region.view(size, size)
    return grid.flatten()

# הגדרת הפרמוטציות לפי המאמר (8x8 נמוך, 26x26 גבוה)
perm_base = None
perm_low = create_partial_permutation(8)
perm_high = create_partial_permutation(26)

# הכנת ה-DataLoaders החדשים
batch_size = 64
loader_base = DataLoader(PermutedMNIST(train_dataset, perm_base), batch_size=batch_size, shuffle=True)
loader_low = DataLoader(PermutedMNIST(train_dataset, perm_low), batch_size=batch_size, shuffle=True)
loader_high = DataLoader(PermutedMNIST(train_dataset, perm_high), batch_size=batch_size, shuffle=True)

# 2. אימון רשתות נפרדות לכל משימה כדי למצוא את המשקלים האופטימליים ולחשב פישר
def train_and_get_fisher(dataloader, epochs=2):
    model = Net().to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.02, momentum=0.9)
    model.train()

    # אימון קצר כדי להתכנס לפתרון של המשימה
    for epoch in range(epochs):
        for data, target in dataloader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            optimizer.step()

    # שימוש בפונקציית הפישר שכבר מוגדרת בקוד שלך
    return compute_fisher(model, dataloader)

print("Training models to compute Fisher matrices for overlap graph...")
fisher_base = train_and_get_fisher(loader_base)
fisher_low = train_and_get_fisher(loader_low)
fisher_high = train_and_get_fisher(loader_high)

# 3. חישוב מרחק Fréchet (חפיפה)
def calculate_overlap(f1, f2, layer_name):
    # לקיחת המשקלים של השכבה (ללא הבייס)
    v1 = f1[layer_name + '.weight'].flatten()
    v2 = f2[layer_name + '.weight'].flatten()

    # נרמול ל-unit trace
    v1_hat = v1 / v1.sum()
    v2_hat = v2 / v2.sum()

    # חישוב d^2 לפי נספח 4.3 במאמר
    d2 = 0.5 * torch.sum((torch.sqrt(v1_hat) - torch.sqrt(v2_hat))**2)

    # overlap = 1 - d^2
    return 1.0 - d2.item()

# השכבות שקיימות ברשת שלך (fc1, fc2, fc3)
layers = ['fc1', 'fc2', 'fc3']
overlap_low = [calculate_overlap(fisher_base, fisher_low, l) for l in layers]
overlap_high = [calculate_overlap(fisher_base, fisher_high, l) for l in layers]

# 4. שרטוט הגרף (מקביל ל-Fig 2C)
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(layers)+1), overlap_low, linestyle='-.', marker='.', color='grey', label='low % permutation')
plt.plot(range(1, len(layers)+1), overlap_high, linestyle='--', marker='.', color='black', label='high % permutation')

plt.xlabel('Layer depth', fontsize=12)
plt.ylabel('Overlap in Fisher', fontsize=12)
plt.xticks(range(1, len(layers)+1))
plt.ylim(0, 1.05)

# עיצוב מינימליסטי בדומה למאמר
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.legend(loc='lower right')
plt.title('Fisher Overlap vs Depth (Fig 2C replication)')
plt.show()